# 04 · Optional materials-science extension (UCI)

**ML Science Lab · hands-on notebook**

**Optional extension:** predict concrete compressive strength (MPa) from proportions of ingredients and sample age. Public UCI data with 1,030 rows and 8 features, CC BY 4.0.

Source: Yeh (1998), UCI Concrete Compressive Strength, DOI https://doi.org/10.24432/C5PK67. **Requires an internet connection to UCI on the first run**. For offline/spotty networks, download and prepare this dataset before class.

In [ ]:
%pip -q install "ucimlrepo" "flaml[automl]>=2,<3"

In [ ]:
from IPython.display import display
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from flaml import AutoML
import numpy as np

concrete = fetch_ucirepo(id=165)
X = concrete.data.features
y = concrete.data.targets.iloc[:, 0]
print('Shape:', X.shape)
display(X.head())
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Baseline and AutoML

The goal is not just to improve error: think about whether a random split would be defensible for repeated mixtures, laboratory batches, concrete curing processes, or experiments from different facilities.

In [ ]:
baseline = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=2)
cv = KFold(n_splits=3, shuffle=True, random_state=42)
cv_rmse = -cross_val_score(baseline, X_dev, y_dev, scoring='neg_root_mean_squared_error', cv=cv)
print('Baseline CV RMSE (MPa):', cv_rmse.mean())

automl = AutoML()
automl.fit(X_train=X_dev, y_train=y_dev, task='regression', metric='rmse',
           eval_method='cv', n_splits=3, time_budget=90,
           estimator_list=['rf', 'extra_tree', 'lgbm'], seed=42, n_jobs=2, verbose=0)
print('AutoML selected estimator:', automl.best_estimator)
print('AutoML internal CV RMSE (MPa):', automl.best_loss)

# One-time test evaluation, after comparing and selecting on CV only
baseline.fit(X_dev, y_dev)
for name, estimator in [('Baseline RF', baseline), ('FLAML', automl)]:
    y_pred = estimator.predict(X_test)
    print(name, 'test RMSE (MPa):', np.sqrt(mean_squared_error(y_test, y_pred)),
          'test R²:', r2_score(y_test, y_pred))

**Exercise:** Which physical input features matter? Why do feature importances from a predictive model not automatically establish causal effects? How would you test robustness across experimental laboratories?